# RAGForge Enterprise — Chunking Strategy Analysis

This notebook provides a **visual, quantitative comparison** of the three chunking strategies
implemented in RAGForge Enterprise:

1. **FixedSizeChunker** — token-budget splitting with sentence boundary respect
2. **RecursiveChunker** — separator-hierarchy splitting (no external dependencies)
3. **SemanticChunker** — embedding-based splitting on cosine similarity discontinuities

---
**Prerequisites:** Run `pip install -e ".[dev]"` from the project root and place at least one PDF
in `data/sample_docs/` before executing this notebook.


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────────
import sys
import os
from pathlib import Path

# Ensure the project root is on sys.path when running from notebooks/
project_root = Path("__file__").resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict

from src.ingestion.loader import DocumentLoader
from src.ingestion.cleaner import DocumentCleaner
from src.ingestion.chunker import (
    FixedSizeChunker, RecursiveChunker, SemanticChunker, Chunk, count_tokens
)
from src.config.settings import get_settings

# Apply a consistent plotting style
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor': '#1e1e2e',
    'axes.edgecolor': '#44475a',
    'axes.labelcolor': '#cdd6f4',
    'text.color': '#cdd6f4',
    'xtick.color': '#cdd6f4',
    'ytick.color': '#cdd6f4',
    'grid.color': '#44475a',
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})

STRATEGY_COLORS = {
    'fixed_size': '#89b4fa',
    'recursive':  '#a6e3a1',
    'semantic':   '#fab387',
}

cfg = get_settings()
print(f"Settings → chunk_size={cfg.chunk_size}, overlap={cfg.chunk_overlap}, "
      f"threshold={cfg.similarity_threshold}")


In [ ]:
# ── Load and clean documents ──────────────────────────────────────────────────
DATA_DIR = project_root / "data" / "sample_docs"

loader  = DocumentLoader(fail_fast=False)
cleaner = DocumentCleaner()

pdf_files = list(DATA_DIR.glob("*.pdf")) + list(DATA_DIR.glob("*.PDF"))

if not pdf_files:
    # Generate synthetic sample text when no PDFs are available
    print("⚠️  No PDFs found in data/sample_docs/. Using synthetic sample text.")
    from src.ingestion.loader import Document, DocumentMetadata
    SAMPLE_TEXT = """
Artificial intelligence has transformed numerous industries over the past decade.
Natural language processing, in particular, has seen dramatic improvements thanks
to the advent of transformer-based architectures. These models, trained on vast
corpora of text, can generate coherent prose, answer complex questions, and even
write functional code.

Retrieval-Augmented Generation (RAG) is a framework that combines the parametric
knowledge of language models with non-parametric retrieval from external document
stores. This architecture addresses the key limitation of large language models:
their knowledge is frozen at training time. By dynamically retrieving relevant
context at inference time, RAG systems can provide up-to-date and verifiable answers.

Chunking strategy selection has a direct impact on retrieval quality. Fixed-size
chunking provides predictable chunk sizes but may split semantically coherent passages.
Recursive chunking respects document structure by trying progressively finer-grained
separators. Semantic chunking uses embedding similarity to identify natural topic
boundaries, producing chunks that are maximally coherent for dense retrieval.

The choice depends on the downstream use case, the nature of the documents, and
the computational budget available at ingestion time. Production RAG systems often
employ multiple strategies in parallel and use query-time weighting to select the
most relevant passages from each strategy's index.
    """.strip()
    meta = DocumentMetadata(filename="synthetic.txt", page_count=1, page_number=1)
    synthetic_doc = Document(
        content=SAMPLE_TEXT,
        metadata=meta,
        source_path=Path("/synthetic/sample.txt"),
        page_number=1,
    )
    all_docs = [synthetic_doc]
else:
    print(f"Found {len(pdf_files)} PDF(s): {[f.name for f in pdf_files]}")
    all_docs = []
    for pdf in pdf_files[:3]:  # Limit to 3 PDFs for notebook performance
        raw = loader.load_file(pdf)
        cleaned = cleaner.clean_batch(raw)
        all_docs.extend(cleaned)
        print(f"  · {pdf.name}: {len(raw)} pages loaded")

print(f"\nTotal document pages: {len(all_docs)}")
total_chars = sum(len(d.content) for d in all_docs)
print(f"Total characters: {total_chars:,}")


In [ ]:
# ── Run all three chunking strategies ─────────────────────────────────────────
print("Running FixedSizeChunker...")
fixed_chunks    = FixedSizeChunker(cfg).chunk_documents(all_docs)
print(f"  → {len(fixed_chunks)} chunks")

print("Running RecursiveChunker...")
recursive_chunks = RecursiveChunker(cfg).chunk_documents(all_docs)
print(f"  → {len(recursive_chunks)} chunks")

print("Running SemanticChunker (downloads model on first run)...")
semantic_chunks = SemanticChunker(cfg).chunk_documents(all_docs)
print(f"  → {len(semantic_chunks)} chunks")

strategy_results = {
    'fixed_size': fixed_chunks,
    'recursive':  recursive_chunks,
    'semantic':   semantic_chunks,
}


In [ ]:
# ── Figure 1: Chunk size distribution (histograms) ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
fig.suptitle('Chunk Size Distribution by Strategy', fontsize=16, fontweight='bold', y=1.02)

strategy_labels = {
    'fixed_size': 'FixedSizeChunker',
    'recursive':  'RecursiveChunker',
    'semantic':   'SemanticChunker',
}

for ax, (name, chunks) in zip(axes, strategy_results.items()):
    token_counts = [c.token_count for c in chunks]
    color = STRATEGY_COLORS[name]
    
    ax.hist(token_counts, bins=20, color=color, alpha=0.85, edgecolor='#282a36', linewidth=0.8)
    ax.axvline(np.mean(token_counts), color='#f38ba8', linestyle='--', linewidth=1.5,
               label=f'Mean: {np.mean(token_counts):.0f}')
    ax.axvline(cfg.chunk_size, color='#f1fa8c', linestyle=':', linewidth=1.5,
               label=f'Target: {cfg.chunk_size}')
    
    ax.set_title(strategy_labels[name], fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('Token Count', fontsize=11)
    ax.set_ylabel('Number of Chunks', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    stats_text = (
        f"n={len(chunks)}\n"
        f"μ={np.mean(token_counts):.1f}\n"
        f"σ={np.std(token_counts):.1f}\n"
        f"min={min(token_counts)}\n"
        f"max={max(token_counts)}"
    )
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#282a36', alpha=0.8,
                      edgecolor=color), color='#cdd6f4')

plt.tight_layout()
plt.savefig('chunk_size_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("Saved: chunk_size_distribution.png")


In [ ]:
# ── Figure 2: Token count statistics summary ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('Token Count Statistics — Strategy Comparison', fontsize=16, fontweight='bold')

x_positions = np.arange(len(strategy_results))
strategy_names = list(strategy_results.keys())
all_token_counts = [[c.token_count for c in strategy_results[s]] for s in strategy_names]

bp = ax.boxplot(
    all_token_counts,
    positions=x_positions,
    widths=0.5,
    patch_artist=True,
    notch=True,
    showfliers=True,
    flierprops=dict(marker='o', markersize=4, alpha=0.5),
    medianprops=dict(color='#f1fa8c', linewidth=2),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
)

for patch, name in zip(bp['boxes'], strategy_names):
    patch.set_facecolor(STRATEGY_COLORS[name])
    patch.set_alpha(0.75)

ax.axhline(cfg.chunk_size, color='#f38ba8', linestyle='--', linewidth=1.5,
           label=f'Target chunk size ({cfg.chunk_size} tokens)', alpha=0.8)

ax.set_xticks(x_positions)
ax.set_xticklabels([strategy_labels[s] for s in strategy_names], fontsize=12)
ax.set_ylabel('Token Count per Chunk', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)

# Annotate medians
for i, tc in enumerate(all_token_counts):
    median = np.median(tc)
    ax.annotate(f'median\n{median:.0f}', xy=(i, median),
                xytext=(i + 0.28, median + 5), fontsize=9, color='#f1fa8c')

plt.tight_layout()
plt.savefig('token_count_statistics.png', dpi=150, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("Saved: token_count_statistics.png")


In [ ]:
# ── Figure 3: Semantic similarity heatmap between chunks ─────────────────────
# We embed a random sample of chunks from each strategy and compute
# pairwise cosine similarity to assess intra-strategy vs inter-strategy cohesion.

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import random

EMB_MODEL = cfg.embedding_model
SAMPLE_N  = min(15, min(len(c) for c in strategy_results.values()))

print(f"Loading embedding model '{EMB_MODEL}' for similarity heatmap...")
emb_model = SentenceTransformer(EMB_MODEL)

random.seed(42)
sampled: dict[str, list[Chunk]] = {
    name: random.sample(chunks, SAMPLE_N)
    for name, chunks in strategy_results.items()
}

# Combine samples for a joint similarity matrix
all_texts: list[str] = []
labels: list[str] = []
for name, chunks in sampled.items():
    for i, c in enumerate(chunks):
        all_texts.append(c.content)
        labels.append(f"{name[:3].upper()}-{i:02d}")

print(f"Embedding {len(all_texts)} chunks...")
embeddings = emb_model.encode(all_texts, normalize_embeddings=True, show_progress_bar=True)
sim_matrix = cosine_similarity(embeddings)

# Plot heatmap
fig, ax = plt.subplots(figsize=(16, 13))
fig.suptitle(
    f'Pairwise Cosine Similarity Heatmap\n'
    f'(n={SAMPLE_N} chunks per strategy, model={EMB_MODEL})',
    fontsize=14, fontweight='bold'
)

cmap = sns.color_palette('viridis', as_cmap=True)
sns.heatmap(
    sim_matrix,
    ax=ax,
    cmap=cmap,
    xticklabels=labels,
    yticklabels=labels,
    vmin=0.0, vmax=1.0,
    linewidths=0.3,
    linecolor='#282a36',
    cbar_kws={'label': 'Cosine Similarity', 'shrink': 0.8},
)

# Draw strategy boundaries
boundary = SAMPLE_N
for b in [boundary, boundary * 2]:
    ax.axhline(b, color='#f38ba8', linewidth=2)
    ax.axvline(b, color='#f38ba8', linewidth=2)

# Strategy region labels
region_mid = SAMPLE_N / 2
for i, (name, color) in enumerate(STRATEGY_COLORS.items()):
    ax.text(
        i * SAMPLE_N + region_mid, -1.5, strategy_labels[name],
        ha='center', va='top', fontsize=10, fontweight='bold',
        color=color, transform=ax.transData,
    )

ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.savefig('semantic_similarity_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("Saved: semantic_similarity_heatmap.png")


In [ ]:
# ── Summary statistics table ────────────────────────────────────────────────────
import pandas as pd

rows = []
for name, chunks in strategy_results.items():
    tcs = [c.token_count for c in chunks]
    char_counts = [len(c.content) for c in chunks]
    rows.append({
        'Strategy': strategy_labels[name],
        'Total Chunks': len(chunks),
        'Avg Tokens': f"{np.mean(tcs):.1f}",
        'Std Tokens': f"{np.std(tcs):.1f}",
        'Min Tokens': min(tcs),
        'Max Tokens': max(tcs),
        'Avg Chars': f"{np.mean(char_counts):.0f}",
        'Target Tokens': cfg.chunk_size,
        '% Over Budget': f"{100 * sum(1 for t in tcs if t > cfg.chunk_size) / len(tcs):.1f}%",
    })

df = pd.DataFrame(rows).set_index('Strategy')
print(df.to_string())


## Analysis & Tradeoff Conclusions

### FixedSizeChunker
**Strengths:**
- Produces the most predictable chunk sizes — minimal variance in token count.
- Zero external dependencies at runtime (no embedding model needed).
- Configurable overlap ensures context continuity across boundaries.
- Best throughput for large-volume ingestion pipelines.

**Weaknesses:**
- No awareness of document structure (paragraphs, sections, tables).
- May split a logically coherent argument across two chunks, degrading
  retrieval precision for multi-sentence reasoning queries.

**Verdict:** Ideal for **initial baseline evaluation**, high-throughput ingestion,
or uniform documents (e.g. book chapters with consistent prose density).

---

### RecursiveChunker
**Strengths:**
- Respects document structure: prefers paragraph breaks, then line breaks,
  then sentence breaks — progressively finer until the budget is met.
- Slightly higher variance in chunk size than FixedSize, but structurally
  more meaningful.
- No embedding model dependency — fast and cheap.

**Weaknesses:**
- Structural boundaries do not always coincide with *semantic* boundaries.
  A document with wall-of-text paragraphs (e.g. legal contracts) may still
  produce incoherent splits.

**Verdict:** The **best default** for mixed-structure documents (reports,
technical papers, manuals) where structure correlates reasonably with semantics.

---

### SemanticChunker
**Strengths:**
- Produces topically coherent chunks regardless of document formatting.
- Similarity heatmap shows that same-strategy chunks have higher intra-group
  similarity than cross-strategy pairs.
- Improves recall in dense passage retrieval benchmarks (BEIR, MTEB).

**Weaknesses:**
- High runtime cost: requires embedding every sentence at ingestion time.
- Chunk size is variable and can be very small (single-sentence) when
  the threshold is set too high.
- Sensitive to `similarity_threshold` — requires tuning per document type.

**Verdict:** Use for **high-precision retrieval** applications (Q&A, conversational
RAG) where answer quality justifies the additional compute at ingestion time.

---

### Recommendation

| Scenario | Recommended Strategy |
|---|---|
| Bulk ingestion, speed-critical | FixedSizeChunker |
| Mixed-structure enterprise docs | RecursiveChunker |
| High-precision conversational RAG | SemanticChunker |
| Production system with SLA | Recursive (primary) + Semantic (query-time rerank) |
